# Feature Fusion: WavLM Embeddings + Text Features → Single XGBoost

Instead of separate models with weighted voting, concatenate all features
into one vector and let a single XGBoost see everything at once.

- WavLM embeddings (768-dim)
- Text + Pause + Prosodic features (41)
- Total: ~809 features → 1 XGBoost

In [ ]:
import os
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix
)
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
import xgboost as xgb
import joblib

FEATURES_CSV = "features_company.csv"
WAVLM_CSV = "features_wavlm.csv"  # generated by wavlm_features.ipynb

print(f"Text features:  {FEATURES_CSV}")
print(f"WavLM features: {WAVLM_CSV}")

## 1. Load & Concatenate Features

In [ ]:
df_text = pd.read_csv(FEATURES_CSV)
df_text = df_text[df_text["label_int"].isin([0, 1])].reset_index(drop=True)

df_wavlm = pd.read_csv(WAVLM_CSV)

assert len(df_text) == len(df_wavlm), f"Row mismatch: text={len(df_text)}, wavlm={len(df_wavlm)}"
print(f"Loaded {len(df_text)} samples")

y = df_text["label_int"].values

# ── Text features (41) ────────────────────────────────────
TEXT_FEATURES = [
    "filler_rate", "filler_count", "repetition_rate", "repair_rate",
    "ttr", "mattr", "complex_word_rate", "avg_word_length",
    "n_words", "n_unique_words",
    "avg_sentence_length", "std_sentence_length", "fragment_rate", "n_sentences",
    "self_ref_rate", "discourse_marker_rate", "hedge_rate",
    "noun_rate", "verb_rate", "adj_rate",
]
PAUSE_FEATURES = [
    "pause_mean", "pause_std", "pause_median", "pause_skew",
    "long_pause_rate", "pause_ratio", "n_pauses", "pause_regularity",
    "pause_before_content_ratio", "pause_before_function_ratio",
    "mid_phrase_pause_rate", "words_per_sec", "articulation_rate",
]
PROSODIC_FEATURES = [
    "f0_mean", "f0_std", "f0_range", "f0_skew", "f0_slope",
    "energy_mean", "energy_std", "speaking_rate_std",
]
TEXT_ONLY = TEXT_FEATURES + PAUSE_FEATURES + PROSODIC_FEATURES
text_cols = [c for c in TEXT_ONLY if c in df_text.columns]

# ── WavLM embedding columns ───────────────────────────────
wavlm_cols = [c for c in df_wavlm.columns if c.startswith("wavlm_")]

print(f"Text features:  {len(text_cols)}")
print(f"WavLM features: {len(wavlm_cols)}")
print(f"Total fused:    {len(text_cols) + len(wavlm_cols)}")

# ── Concatenate ───────────────────────────────────────────
X_text = df_text[text_cols].fillna(0).values
X_wavlm = df_wavlm[wavlm_cols].fillna(0).values
X_fused = np.hstack([X_text, X_wavlm])
all_cols = text_cols + wavlm_cols

print(f"\nFused matrix: {X_fused.shape}")
print(f"  Cheating: {y.sum()}, Not cheating: {(y==0).sum()}")

## 2. Train/Test Split + Train Fused XGBoost

In [ ]:
idx_train, idx_test = train_test_split(
    np.arange(len(y)), test_size=0.2, random_state=42, stratify=y
)
y_train, y_test = y[idx_train], y[idx_test]

print(f"Train: {len(y_train)} (cheating={y_train.sum()})")
print(f"Test:  {len(y_test)} (cheating={y_test.sum()})")

if "audio_batch" in df_text.columns:
    for name, idxs in [("Train", idx_train), ("Test", idx_test)]:
        counts = df_text.iloc[idxs]["audio_batch"].value_counts().to_dict()
        print(f"  {name} batches: {counts}")

# Scale
scaler = StandardScaler()
X_train = scaler.fit_transform(X_fused[idx_train])
X_test = scaler.transform(X_fused[idx_test])

# Train
model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.2,  # low — 809 features, prevent overfitting
    min_child_weight=3,
    scale_pos_weight=(y_train == 0).sum() / max((y_train == 1).sum(), 1),
    eval_metric="logloss",
    early_stopping_rounds=30,
    random_state=42,
)

model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=20,
)

proba_test = model.predict_proba(X_test)[:, 1]
preds_test = (proba_test >= 0.5).astype(int)

print(f"\n--- Fused XGBoost (test set) ---")
print(f"Accuracy: {accuracy_score(y_test, preds_test):.4f}")
print(f"F1:       {f1_score(y_test, preds_test, zero_division=0):.4f}")
print(classification_report(y_test, preds_test, target_names=["not cheating", "cheating"]))

## 3. Feature Importance — Which features matter most?

In [ ]:
importances = model.feature_importances_
feat_imp = sorted(zip(all_cols, importances), key=lambda x: x[1], reverse=True)

# Separate text vs wavlm importance
text_imp = sum(imp for name, imp in feat_imp if not name.startswith("wavlm_"))
wavlm_imp = sum(imp for name, imp in feat_imp if name.startswith("wavlm_"))
total_imp = text_imp + wavlm_imp

print(f"Feature importance share:")
print(f"  Text features (41):    {text_imp/total_imp*100:.1f}%")
print(f"  WavLM embeddings:      {wavlm_imp/total_imp*100:.1f}%")

print(f"\nTop 20 features:")
for name, imp in feat_imp[:20]:
    group = "WAVLM" if name.startswith("wavlm_") else "TEXT"
    print(f"  [{group:5s}] {name:>30s}  {imp:.4f}")

## 4. Threshold Sweep

In [ ]:
thresholds = np.arange(0.10, 0.91, 0.05)
rows_t = []
for t in thresholds:
    preds = (proba_test >= t).astype(int)
    rows_t.append({
        "threshold": round(t, 2),
        "precision": round(precision_score(y_test, preds, zero_division=0), 4),
        "recall": round(recall_score(y_test, preds, zero_division=0), 4),
        "f1": round(f1_score(y_test, preds, zero_division=0), 4),
        "flagged": int(preds.sum()),
        "missed": int(((y_test == 1) & (preds == 0)).sum()),
        "false_alarms": int(((y_test == 0) & (preds == 1)).sum()),
    })

thresh_df = pd.DataFrame(rows_t)
best_row = thresh_df.loc[thresh_df["f1"].idxmax()]

print(f"Threshold sweep (fused model)")
print(f"{'thresh':>7s} {'prec':>7s} {'recall':>7s} {'f1':>7s} {'flagged':>8s} {'missed':>7s} {'false_alarm':>11s}")
print("-" * 62)
for _, r in thresh_df.iterrows():
    marker = " <-- best F1" if r["threshold"] == best_row["threshold"] else ""
    print(f"  {r['threshold']:.2f}   {r['precision']:.4f}  {r['recall']:.4f}  {r['f1']:.4f}  {r['flagged']:>6d}  {r['missed']:>6d}  {r['false_alarms']:>6d}{marker}")

CHOSEN_THRESHOLD = best_row["threshold"]
print(f"\nBest F1 at threshold={CHOSEN_THRESHOLD:.2f}: F1={best_row['f1']:.4f}")

## 5. 5-Fold Cross-Validation

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_results = []

for fold, (tr_idx, te_idx) in enumerate(skf.split(X_fused, y), 1):
    sc = StandardScaler()
    X_tr = sc.fit_transform(X_fused[tr_idx])
    X_te = sc.transform(X_fused[te_idx])

    m = xgb.XGBClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.2,
        min_child_weight=3,
        scale_pos_weight=(y[tr_idx]==0).sum() / max((y[tr_idx]==1).sum(), 1),
        eval_metric="logloss", random_state=42,
    )
    m.fit(X_tr, y[tr_idx], verbose=False)

    prob = m.predict_proba(X_te)[:, 1]
    preds = (prob >= CHOSEN_THRESHOLD).astype(int)

    f = f1_score(y[te_idx], preds, zero_division=0)
    p = precision_score(y[te_idx], preds, zero_division=0)
    r = recall_score(y[te_idx], preds, zero_division=0)
    fold_results.append({"fold": fold, "f1": f, "precision": p, "recall": r})
    print(f"  Fold {fold}: F1={f:.4f}  Prec={p:.4f}  Rec={r:.4f}")

fold_df = pd.DataFrame(fold_results)
print(f"\n  Mean:  F1={fold_df['f1'].mean():.4f} +/- {fold_df['f1'].std():.4f}")
print(f"         Prec={fold_df['precision'].mean():.4f} +/- {fold_df['precision'].std():.4f}")
print(f"         Rec={fold_df['recall'].mean():.4f} +/- {fold_df['recall'].std():.4f}")

## 6. Save

In [ ]:
SAVE_DIR = "checkpoints_fused"
os.makedirs(SAVE_DIR, exist_ok=True)

model.save_model(f"{SAVE_DIR}/xgboost_fused.json")
joblib.dump(scaler, f"{SAVE_DIR}/scaler_fused.pkl")

config = {
    "feature_columns": all_cols,
    "n_text_features": len(text_cols),
    "n_wavlm_features": len(wavlm_cols),
    "threshold": float(CHOSEN_THRESHOLD),
    "test_f1": round(float(best_row["f1"]), 4),
    "cv_f1_mean": round(fold_df["f1"].mean(), 4),
    "text_importance_pct": round(text_imp/total_imp*100, 1),
    "wavlm_importance_pct": round(wavlm_imp/total_imp*100, 1),
    "n_train": len(idx_train),
    "n_test": len(idx_test),
}
with open(f"{SAVE_DIR}/results_fused.json", "w") as f:
    json.dump(config, f, indent=2)

print(f"Saved to {SAVE_DIR}/:")
print(f"  xgboost_fused.json  - fused model")
print(f"  scaler_fused.pkl    - scaler")
print(f"  results_fused.json  - config + results")

## 7. Predict on All Data + Error Analysis

In [ ]:
X_all_scaled = scaler.transform(X_fused)
all_proba = model.predict_proba(X_all_scaled)[:, 1]

df_text["fused_score"] = all_proba
df_text["pred_label"] = (all_proba >= CHOSEN_THRESHOLD).astype(int)
df_text["pred_label_str"] = df_text["pred_label"].map({1: "cheating", 0: "not cheating"})

print(f"Predictions ({len(df_text)} files):")
print(f"  Cheating:     {(df_text['pred_label']==1).sum()}")
print(f"  Not cheating: {(df_text['pred_label']==0).sum()}")
print(f"  Threshold: {CHOSEN_THRESHOLD:.2f}")

# Error analysis
wrong = df_text[df_text["label_int"] != df_text["pred_label"]]
fp = wrong[wrong["pred_label"] == 1]
fn = wrong[wrong["pred_label"] == 0]
print(f"\nMisclassifications: {len(wrong)} ({len(fp)} FP, {len(fn)} FN)")

if len(wrong) > 0:
    show_cols = ["filename", "label_int", "pred_label_str", "fused_score"]
    if "audio_batch" in wrong.columns:
        show_cols.insert(1, "audio_batch")
    show_cols = [c for c in show_cols if c in wrong.columns]
    print(wrong[show_cols].to_string(index=False))

# Save
out_cols = ["filename", "filepath", "label_int", "pred_label_str", "fused_score"]
if "audio_batch" in df_text.columns:
    out_cols.insert(2, "audio_batch")
out_cols = [c for c in out_cols if c in df_text.columns]
df_text[out_cols].to_csv("predictions_fused.csv", index=False)
print(f"\nSaved: predictions_fused.csv")